<a href="https://colab.research.google.com/github/muhammedshahidsds25-collab/Sentiment-Analysis-Product-Reviews/blob/main/notebooks/02_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment Analysis of Product Reviews
## Notebook 2: Feature Engineering, Selection, Model Training & Comparison

## Required Imports

In [1]:
import os
import re
import time
import warnings
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────
RANDOM_STATE   = 42
TEST_SIZE      = 0.2
TOP_K_CHI2     = 500
TOP_K_MI       = 300
TFIDF_MAX_FEAT = 15000

NEGATION_WORDS = {
    "not","no","never","none","nobody","nothing","neither","nor","nowhere",
    "hardly","barely","scarcely","doesn't","didn't","don't","won't","wouldn't",
    "couldn't","shouldn't","isn't","aren't","wasn't","weren't","haven't","hasn't",
    "hadn't","can't","cannot","shan't","mustn't","needn't","daren't","without"
}

SENTIMENT_ORDER = ["Negative", "Neutral", "Positive"]

In [4]:
df = pd.read_csv(
    "/content/archive (1).zip",
    on_bad_lines="skip",
    engine="python",
    encoding="utf-8"
)

# 🔍 See actual columns
print("Columns:", df.columns)

# 🔧 Try common column names automatically
possible_text_cols = ["Review Text", "review_text", "review", "Text", "content"]
possible_rating_cols = ["Rating", "rating", "Score", "score"]

text_col = None
rating_col = None

for col in possible_text_cols:
    if col in df.columns:
        text_col = col
        break

for col in possible_rating_cols:
    if col in df.columns:
        rating_col = col
        break

# 🚨 Safety check
if text_col is None or rating_col is None:
    raise ValueError("Could not find required columns. Check dataset manually.")

# Rename
df = df[[text_col, rating_col]].copy()
df.columns = ["review_text", "rating"]

# Continue processing
df["rating"] = df["rating"].astype(str).str.extract(r'(\d)').astype(float)
df = df.dropna(subset=["review_text", "rating"])
df = df[df["rating"].between(1, 5)]
df["rating"] = df["rating"].astype(int)

SENTIMENT_MAP = {1: "Negative", 2: "Negative", 3: "Neutral", 4: "Positive", 5: "Positive"}
df["sentiment"] = df["rating"].map(SENTIMENT_MAP)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess(df):
    df["cleaned_text"] = df["review_text"].apply(clean_text)
    return df


df = preprocess(df)

Columns: Index(['Reviewer Name', 'Profile Link', 'Country', 'Review Count',
       'Review Date', 'Rating', 'Review Title', 'Review Text',
       'Date of Experience'],
      dtype='object')


## SECTION 3 – FEATURE ENGINEERING

In [5]:
def extract_linguistic_features(df: pd.DataFrame) -> np.ndarray:
    """
    Compute 4 hand-crafted linguistic features per review:
      1. sentence_length      – number of words in cleaned text
      2. punctuation_density  – punctuation chars / total chars (raw text)
      3. avg_word_length      – mean character length of words
      4. has_negation         – 1 if any negation word present, else 0
    """
    feats = pd.DataFrame()

    # 1. Sentence length (word count of cleaned text)
    feats["sentence_length"] = df["cleaned_text"].str.split().str.len().fillna(0)

    # 2. Punctuation density (raw text)
    def punct_density(s):
        if not isinstance(s, str) or len(s) == 0:
            return 0.0
        punct = sum(1 for c in s if c in "!?,.:;\"'()[]{}\u2026-")
        return punct / len(s)

    feats["punctuation_density"] = df["review_text"].apply(punct_density)

    # 3. Average word length (cleaned text)
    def avg_word_len(s):
        words = s.split() if isinstance(s, str) else []
        if not words:
            return 0.0
        return np.mean([len(w) for w in words])

    feats["avg_word_length"] = df["cleaned_text"].apply(avg_word_len)

    # 4. Presence of negation words (raw text)
    def has_negation(s):
        if not isinstance(s, str):
            return 0
        tokens = set(re.sub(r"[^a-z\s']", " ", s.lower()).split())
        return int(bool(tokens & NEGATION_WORDS))

    feats["has_negation"] = df["review_text"].apply(has_negation)

    return feats.values.astype(np.float32)


def build_features(df: pd.DataFrame):
    """
    Build combined feature matrix:
      - TF-IDF with unigrams + bigrams (sparse)
      - Linguistic features (dense → sparse)

    Returns
    -------
    X_combined : scipy sparse matrix
    tfidf      : fitted TfidfVectorizer
    """
    print(f"\n{'='*60}")
    print("SECTION 3: FEATURE ENGINEERING")
    print(f"{'='*60}")

    # ── TF-IDF ────────────────────────────────────────────
    print(f"  ► Fitting TF-IDF (max_features={TFIDF_MAX_FEAT}, "
          f"ngram_range=(1,2))…")
    tfidf = TfidfVectorizer(
        max_features=TFIDF_MAX_FEAT,
        ngram_range=(1, 2),
        sublinear_tf=True,          # log(1+tf)
        min_df=3,
        max_df=0.95,
        strip_accents="unicode",
        analyzer="word",
        token_pattern=r"\b[a-z]{2,}\b"
    )
    X_tfidf = tfidf.fit_transform(df["cleaned_text"])
    print(f"  ► TF-IDF matrix shape  : {X_tfidf.shape}")

    # ── Linguistic features ──────────────────────────────
    print("  ► Extracting linguistic features…")
    X_ling = extract_linguistic_features(df)
    X_ling_sparse = csr_matrix(X_ling)
    print(f"  ► Linguistic feats shape: {X_ling.shape}")

    # ── Combine ──────────────────────────────────────────
    X_combined = hstack([X_tfidf, X_ling_sparse])
    print(f"  ► Combined feature shape: {X_combined.shape}")

    return X_combined, tfidf

## SECTION 4 – FEATURE SELECTION

In [6]:
def select_features(X, y, method: str = "chi2", k: int = TOP_K_CHI2):
    """
    Apply univariate feature selection.

    Parameters
    ----------
    X      : sparse feature matrix
    y      : encoded label array
    method : 'chi2' or 'mutual_info'
    k      : number of top features to keep

    Returns
    -------
    X_selected : reduced sparse matrix
    selector   : fitted SelectKBest object
    """
    print(f"\n  ► Feature selection — method='{method}', k={k}")

    score_func = chi2 if method == "chi2" else mutual_info_classif
    selector = SelectKBest(score_func=score_func, k=min(k, X.shape[1]))
    X_selected = selector.fit_transform(X, y)
    print(f"    Selected shape: {X_selected.shape}")
    return X_selected, selector

In [7]:
# Example usage — assumes df is loaded and preprocessed from Notebook 1
# X, tfidf = build_features(df)
# le = LabelEncoder()
# le.fit(SENTIMENT_ORDER)
# y = le.transform(df["sentiment"])
#
# print(f"\n{'='*60}")
# print("SECTION 4: FEATURE SELECTION")
# print(f"{'='*60}")
# X_chi2, sel_chi2 = select_features(X, y, method="chi2",        k=TOP_K_CHI2)
# X_mi,   sel_mi   = select_features(X, y, method="mutual_info", k=TOP_K_MI)
# print(f"\n  ► Using Chi2-selected features for model training.")
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X_chi2, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
# )
# print(f"\n  Train size: {X_train.shape[0]:,}   Test size: {X_test.shape[0]:,}")

## SECTION 5 – MODEL TRAINING & EVALUATION

In [8]:
def get_models() -> dict:
    """Return a dictionary of model name → estimator."""
    return {
        "Logistic Regression": LogisticRegression(
    C=1.0,
    max_iter=1000,
    solver="lbfgs",
    class_weight="balanced",
    random_state=RANDOM_STATE
),
        "Multinomial NB": MultinomialNB(alpha=0.5),
        "Linear SVM": CalibratedClassifierCV(
    LinearSVC(
        C=1.0, max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    cv=3
),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=150, learning_rate=0.1, max_depth=4,
            subsample=0.8, random_state=RANDOM_STATE
        ),
    }


def evaluate_model(model, X_train, X_test, y_train, y_test,
                   label_names: list) -> dict:
    """
    Train a model, evaluate on test set, and return a metrics dict.
    """
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = model.predict(X_test)

    metrics = {
        "accuracy"  : accuracy_score(y_test, y_pred),
        "precision" : precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall"    : recall_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_f1"  : f1_score(y_test, y_pred, average="macro", zero_division=0),
        "train_time": train_time,
        "report"    : classification_report(y_test, y_pred,
                                            target_names=label_names,
                                            zero_division=0),
        "conf_matrix": confusion_matrix(y_test, y_pred),
        "y_pred"    : y_pred,
    }
    return metrics


def train_and_evaluate(X_train, X_test, y_train, y_test,
                       label_names: list) -> tuple[dict, dict]:
    """
    Train all models and collect evaluation results.

    Returns
    -------
    results  : dict  { model_name → metrics_dict }
    models   : dict  { model_name → fitted estimator }
    """
    print(f"\n{'='*60}")
    print("SECTION 5: MODEL TRAINING & EVALUATION")
    print(f"{'='*60}")

    models_def = get_models()
    results = {}
    fitted_models = {}

    for name, model in models_def.items():
        print(f"\n  ▷ Training: {name} …")
        metrics = evaluate_model(model, X_train, X_test, y_train, y_test, label_names)
        results[name] = metrics
        fitted_models[name] = model

        print(f"    Accuracy  : {metrics['accuracy']:.4f}")
        print(f"    Precision : {metrics['precision']:.4f}")
        print(f"    Recall    : {metrics['recall']:.4f}")
        print(f"    Macro F1  : {metrics['macro_f1']:.4f}")
        print(f"    Train time: {metrics['train_time']:.2f}s")
        print()
        print("  Classification Report:")
        for line in metrics["report"].splitlines():
            print(f"    {line}")

    return results, fitted_models

## SECTION 6 – MODEL COMPARISON

In [18]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

def compare_models(results: dict) -> pd.DataFrame:
    """
    Build a comparison DataFrame and identify best model by Macro F1.
    """
    print(f"\n{'='*60}")
    print("SECTION 6: MODEL COMPARISON")
    print(f"{'='*60}")

    rows = []
    for name, m in results.items():
        rows.append({
            "Model"     : name,
            "Accuracy"  : round(m["accuracy"], 4),
            "Precision" : round(m["precision"], 4),
            "Recall"    : round(m["recall"], 4),
            "Macro F1"  : round(m["macro_f1"], 4),
            "Train(s)"  : round(m["train_time"], 2),
        })

    cmp_df = pd.DataFrame(rows).sort_values("Macro F1", ascending=False).reset_index(drop=True)
    best   = cmp_df.iloc[0]["Model"]

    print(f"\n  {'\u2500'*65}")
    print(cmp_df.to_string(index=False))
    print(f"  {'\u2500'*65}")
    print(f"\n  \u2605  Best model by Macro F1 \u2192 {best}  "
          f"(F1 = {cmp_df.iloc[0]['Macro F1']:.4f})")

    return cmp_df, best

In [19]:
# Example usage — assumes X_train, X_test, y_train, y_test are defined above
# results, fitted_models = train_and_evaluate(
#     X_train, X_test, y_train, y_test,
#     label_names=list(le.classes_)
# )
# cmp_df, best_model_name = compare_models(results)

In [20]:
# ================================
# RUN FULL PIPELINE
# ================================

# Build features
X, tfidf = build_features(df)

# Encode labels
le = LabelEncoder()
le.fit(SENTIMENT_ORDER)
y = le.transform(df["sentiment"])

# Feature selection
print("\n=== Feature Selection ===")
X_sel, selector = select_features(X, y, method="chi2", k=TOP_K_CHI2)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f"\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

# Train models
results, models = train_and_evaluate(
    X_train, X_test, y_train, y_test,
    label_names=list(le.classes_)
)

# Compare models
cmp_df, best_model = compare_models(results)


print("\n=== FINAL RESULT ===")
print(f"Best Model: {best_model}")
print(f"Macro F1 Score: {cmp_df.iloc[0]['Macro F1']}")


SECTION 3: FEATURE ENGINEERING
  ► Fitting TF-IDF (max_features=15000, ngram_range=(1,2))…
  ► TF-IDF matrix shape  : (21055, 15000)
  ► Extracting linguistic features…
  ► Linguistic feats shape: (21055, 4)
  ► Combined feature shape: (21055, 15004)

=== Feature Selection ===

  ► Feature selection — method='chi2', k=500
    Selected shape: (21055, 500)

Train size: 16844 | Test size: 4211

SECTION 5: MODEL TRAINING & EVALUATION

  ▷ Training: Logistic Regression …
    Accuracy  : 0.7910
    Precision : 0.6370
    Recall    : 0.7028
    Macro F1  : 0.6324
    Train time: 13.36s

  Classification Report:
                  precision    recall  f1-score   support
    
        Negative       0.96      0.80      0.87      2870
         Neutral       0.14      0.50      0.21       177
        Positive       0.82      0.80      0.81      1164
    
        accuracy                           0.79      4211
       macro avg       0.64      0.70      0.63      4211
    weighted avg       0.88  